# 03 - Evaluation of the Atlas Enterprise Assistant

This notebook loads `outputs/eval_report.json` (created by `python -m tests.evaluate`) and shows:

1. Summary numbers
2. Per-case results
3. Charts
4. The cases that failed and why
5. The answers the assistant produced

By default it only **reads** the saved report. To re-run the evaluation from here, set `RUN_EVAL = True` in the optional cell below (needs `GOOGLE_API_KEY` and a built index).

In [ ]:
import json
import os
import pathlib
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Find the repo root whether the notebook runs from the root or from notebooks/
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
REPORT_PATH = ROOT / "outputs" / "eval_report.json"
print("Repo root:", ROOT)
print("Report:", REPORT_PATH, "| exists:", REPORT_PATH.exists())

## Optional: run the evaluation from this notebook

Leave `RUN_EVAL = False` to just read the saved report. The full run takes 15-30 minutes because every question is judged by five DeepEval metrics and the runner pauses between calls to respect API rate limits.

If the default model has no quota left, set the model names below (any model your key can use).

In [ ]:
RUN_EVAL = False          # set to True to re-run the evaluation
LIMIT = None              # for a quick test use e.g. 2
SKIP_METRICS = False      # True = routing + citations only (no judge calls)
MODEL_OVERRIDE = None     # e.g. "gemini-3.1-flash-lite" (used for both assistant and judge)

if RUN_EVAL:
    env = os.environ.copy()
    if MODEL_OVERRIDE:
        env["LLM_MODEL"] = MODEL_OVERRIDE
        env["JUDGE_MODEL"] = MODEL_OVERRIDE
    cmd = [sys.executable, "-m", "tests.evaluate"]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    if SKIP_METRICS:
        cmd += ["--skip-metrics"]
    subprocess.run(cmd, cwd=ROOT, env=env, check=True)
else:
    print("Skipped (RUN_EVAL = False). Using the saved report.")

## 1. Summary

In [ ]:
data = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
summary, results = data["summary"], data["results"]

print(f"Cases evaluated:           {summary['cases']}")
print(f"Cases with runtime errors: {summary['errors']}")
print(f"Routing accuracy:          {summary['routing_accuracy']:.0%}")
print(f"Citation recall:           {summary['citation_recall']:.0%}")
print(f"Citation full-match rate:  {summary['citation_full_match_rate']:.0%}")

metrics_df = pd.DataFrame(summary["metrics"]).T.rename(
    columns={"average_score": "average score", "pass_rate": "pass rate", "evaluated": "cases scored"}
)
display(metrics_df)

lang_df = pd.DataFrame(summary["by_language"]).T
display(lang_df)

> **How to read this:** DeepEval metrics are only computed for cases where the pipeline retrieved context. If the router returns `unknown`, nothing is retrieved and the case has no Faithfulness, Hallucination, or Contextual scores (shown as NaN). A high Faithfulness average therefore describes only the questions the system actually answered.

## 2. Per-case results

In [ ]:
METRICS = ["faithfulness", "answer_relevancy", "hallucination", "contextual_precision", "contextual_recall"]

rows = []
for r in results:
    row = {
        "id": r["id"],
        "lang": r.get("language"),
        "expected": r["expected_domain"],
        "predicted": r.get("predicted_domain"),
        "routing_ok": r["routing_correct"],
        "citation_recall": r["citations"]["recall"],
        "llm_error": r.get("llm_error", False),
    }
    for m in METRICS:
        row[m] = (r.get("metrics", {}).get(m) or {}).get("score")
    rows.append(row)

df = pd.DataFrame(rows)
display(df)

## 3. Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Average DeepEval scores
avg = df[METRICS].mean().dropna()
axes[0].bar([m.replace("_", "\n") for m in avg.index], avg.values, color="#4C78A8")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Average DeepEval score (cases that reached generation)")
for i, v in enumerate(avg.values):
    axes[0].text(i, v + 0.02, f"{v:.2f}", ha="center")

# Routing and citations by language
by_lang = df.groupby("lang").agg(routing=("routing_ok", "mean"), citations=("citation_recall", "mean"))
by_lang.plot(kind="bar", ax=axes[1], rot=0, color=["#F58518", "#54A24B"])
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Routing accuracy and citation recall by language")
axes[1].set_xlabel("language")

plt.tight_layout()
plt.show()

In [ ]:
# Overall: how many cases reached generation vs. were stopped by the router?
counts = df["predicted"].value_counts()
print("Predicted domains:")
print(counts.to_string())

plt.figure(figsize=(5, 4))
df["routing_ok"].map({True: "Correct", False: "Wrong / unknown"}).value_counts().plot(
    kind="bar", rot=0, color=["#54A24B", "#E45756"]
)
plt.title("Routing outcome")
plt.ylabel("cases")
plt.show()

## 4. Failed cases

In [ ]:
failed = [r for r in results if not r["routing_correct"]]
print(f"{len(failed)} of {len(results)} cases were routed incorrectly.\n")
for r in failed:
    print(r["id"], f"[{r.get('language')}]")
    print("  Question: ", r["question"])
    print("  Expected: ", r["expected_domain"], "| Predicted:", r.get("predicted_domain"))
    print("  Missing sources:", r["citations"]["missing"])
    print()

### Reading this section

If the list above is empty, every question was routed to the expected domain. If it is not empty, the usual cause is the keyword-based router in `src/router.py`: a question with no matching keyword ends as `unknown`, the workflow stops before retrieval, and no DeepEval context metrics exist for that case.

First run (English-only keyword lists): routing 40%, Arabic 0 of 3. After the router update (English and Arabic combined): routing 100%.

## 4b. Manual check: do the answers contain the key facts?

In [ ]:
# DeepEval's Hallucination and Faithfulness compare the answer with the *retrieved context*,
# not with ground truth. This heuristic checks that key figures from the reference answers
# appear in the answer text. It only tests that a figure is present, not that it is used correctly.
KEY_FACTS = {
    "EVAL-001": ["15"],
    "EVAL-002": ["14", "5"],
    "EVAL-003": ["90"],
    "EVAL-004": ["vpn.atlas-industries.com", "443"],
    "EVAL-005": ["24"],
    "EVAL-006": ["Microsoft Authenticator"],
    "EVAL-007": ["1,500", "800"],
    "EVAL-008": ["75", "110"],
    "EVAL-009": ["2,000", "15,000"],
    "EVAL-010": ["3 to 5", "5 to 10", "10 to 20"],
}

rows = []
for r in results:
    facts = KEY_FACTS.get(r["id"], [])
    answer = r.get("answer") or ""
    missing = [f for f in facts if f not in answer]
    rows.append({"id": r["id"], "key facts": facts, "missing": missing, "all present": not missing})
facts_df = pd.DataFrame(rows)
display(facts_df)
print(f"Answers containing all key facts: {facts_df['all present'].sum()} of {len(facts_df)}")

Cases with missing facts show where retrieval failed to bring the right passage, even when Hallucination is 1.0. In the latest run these are EVAL-007 (hotel and taxi caps not found) and EVAL-008 (the international per diem was answered from the domestic document). EVAL-009 also contains an extra approval tier that this check cannot detect, which is why the answers should be read, not only scored.

## 5. The assistant's answers

In [ ]:
for r in results:
    print("=" * 80)
    print(r["id"], f"[{r.get('language')}]  routing:", "OK" if r["routing_correct"] else "FAIL")
    print("Q:", r["question"])
    print("A:", (r.get("answer") or "(no answer stored)").strip()[:600])
    if r.get("expected_output"):
        print("Reference:", r["expected_output"][:300])

## 6. Conclusions

**Before and after the router update**

| Metric | Baseline | After router update |
|---|---|---|
| Routing accuracy | 40% | 100% |
| Routing, Arabic | 0 of 3 | 3 of 3 |
| Citation recall | 40% | 95% |
| Faithfulness | 1.00 (4 cases) | 0.93 (10 cases) |
| Answer Relevancy | 0.55 | 0.93 |
| Hallucination | 1.00 (4 cases) | 1.00 (10 cases) |
| Contextual Precision | 0.83 (4 cases) | 0.82 |
| Contextual Recall | 0.75 (4 cases) | 0.82 |

**What the numbers do and do not show**

- Routing is no longer the bottleneck on these 10 questions (the router was extended with vocabulary that partly overlaps the questions, so this may be optimistic).
- Retrieval is now the main weakness: Contextual Precision passes in only 70% of cases, and the two worst answers (EVAL-007 and EVAL-008) are caused by missing or wrong chunks.
- Hallucination = 1.0 does not mean every answer is correct: EVAL-008 states the wrong international per diem because it answered from the domestic document.
- Recommended next steps: tune retrieval (chunking, `MAX_CHUNKS_PER_FILE`, reranking), add an exact-match check for locked facts, and strip the sources block before judging.

Both runs used `gemini-3.1-flash-lite` as assistant and judge because the default model hit its free-tier quota.